In [1]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

load_dotenv()
login(os.getenv("HF_TOKEN"))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


# Basic RAG

In [2]:
#Basic RAG

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

with open("data/knowledge_base.txt") as f:
    docs = f.readlines()

model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = model.encode(docs)

def retrieve(query, k=3):
    query_embedding = model.encode([query])

    scores = cosine_similarity(
        query_embedding,
        doc_embeddings
    )[0]

    idx = np.argsort(scores)[-k:]

    return [docs[i] for i in idx]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

c:\Users\visarati\Desktop\AntiGravity\VS code\AI_agents\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\visarati\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# CRAG


In [3]:
#CRAG

def crag(query):

    query_emb = model.encode([query])

    scores = cosine_similarity(
        query_emb,
        doc_embeddings
    )[0]

    best_score = max(scores)

    docs_found = retrieve(query)

    if best_score < 0.35:
        print("Low confidence")
        print("Performing corrective retrieval")

        docs_found = retrieve(query, k=5)

    return docs_found

# Adaptive RAG (ARAG)


In [4]:
#Adaptive RAG (ARAG)

def adaptive_rag(query):

    keywords = [
        "what",
        "explain",
        "define",
        "tell"
    ]

    if any(word in query.lower()
           for word in keywords):

        print("Using Retrieval")

        return retrieve(query)

    print("No Retrieval Needed")

    return "Answered directly by LLM"

# Self-RAG


In [5]:
#Self-RAG

def self_rag(query):

    docs_found = retrieve(query)

    answer = " ".join(docs_found)

    print("Initial Answer")
    print(answer)

    print("\nCritic Reviewing...")

    if len(answer) < 100:
        answer += "\n\nAdditional context added."

    return answer

## Testing it all

In [10]:
retrieve("What is AI agent?")

['AI engineering combines software and AI.',
 'LangGraph helps build AI agents.\n',
 'Agents use tools to perform tasks.\n']

In [11]:
crag("Explain CRAG")

['CRAG evaluates retrieved documents.\n',
 'CRAG may perform additional retrieval.\n',
 'CRAG stands for Corrective Retrieval Augmented Generation.\n']

In [12]:
adaptive_rag(
 "What is LangGraph?"
)

adaptive_rag(
 "Write a poem about AI"
)

Using Retrieval
No Retrieval Needed


'Answered directly by LLM'

In [13]:
self_rag(
 "What is Self RAG?"
)

Initial Answer
RAG stands for Retrieval Augmented Generation.
 Self-RAG can improve generated responses.
 Self-RAG critiques its own answers.


Critic Reviewing...


'RAG stands for Retrieval Augmented Generation.\n Self-RAG can improve generated responses.\n Self-RAG critiques its own answers.\n'